In [ ]:
from google.colab import drive
import os
import pandas as pd
from sklearn.model_selection import train_test_split
import pickle
import re
from collections import Counter

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Copying 8k to Colabs local SSD
if not os.path.exists("/content/flickr8k.zip"):
  !cp -r "/content/drive/MyDrive/MMRetrieval/flickr8k.zip" /content/

In [ ]:
!unzip "/content/flickr8k.zip" -d "/content/flickr8k"

In [ ]:
# Copying 30k to Colabs local SSD
if not os.path.exists("/content/flickr30k.zip"):
  !cp -r "/content/drive/MyDrive/MMRetrieval/flickr30k.zip" /content/

In [ ]:
!unzip "/content/flickr30k.zip" -d "/content/flickr30k"

In [ ]:
def clean_caption(text):
    text = text.lower()

    # remove punctuation
    text = re.sub(r"[^a-z0-9\s]", "", text)

    # collapse spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
def split_dataset(images,dataset_name):
  train_imgs, temp_imgs = train_test_split(
    images,
    train_size=0.75,
    random_state=42,
    shuffle=True
    )
  val_imgs, test_imgs = train_test_split(
        temp_imgs,
        test_size=0.5,
        random_state=42,
        shuffle=True
    )
  print(f"Train : {len(train_imgs)}")
  print(f"Val   : {len(val_imgs)}")
  print(f"Test  : {len(test_imgs)}")
  split_dict = {
    "train": train_imgs,
    "val": val_imgs,
    "test": test_imgs
    }
  SAVE_DIR = f"/content/drive/MyDrive/MMRetrieval/Preprocessing/{dataset_name}"
  os.makedirs(SAVE_DIR, exist_ok=True)
  with open(os.path.join(SAVE_DIR, f"{dataset_name}_split.pkl"), "wb") as f:
      pickle.dump(split_dict, f)
  return split_dict

In [ ]:
def create_vocab(df,split_dict,dataset_name):
  SAVE_DIR = f"/content/drive/MyDrive/MMRetrieval/Preprocessing/{dataset_name}"
  train_imgs = split_dict["train"]
  val_imgs = split_dict["val"]
  test_imgs = split_dict["test"]

  train_df = df[df["image"].isin(train_imgs)].reset_index(drop=True)
  val_df = df[df["image"].isin(val_imgs)].reset_index(drop=True)
  test_df = df[df["image"].isin(test_imgs)].reset_index(drop=True)

  train_df["caption"] = train_df["caption"].apply(clean_caption)
  val_df["caption"] = val_df["caption"].apply(clean_caption)
  test_df["caption"] = test_df["caption"].apply(clean_caption)
  PAD_TOKEN = "<PAD>"
  UNK_TOKEN = "<UNK>"
  SOS_TOKEN = "<SOS>"
  EOS_TOKEN = "<EOS>"

  counter = Counter()
  for caption in train_df["caption"]:
      counter.update(caption.split())

  MIN_FREQ = 5

  vocab = {
      PAD_TOKEN: 0,
      UNK_TOKEN: 1,
      SOS_TOKEN: 2,
      EOS_TOKEN: 3
  }

  for word, freq in counter.items():
      if freq >= MIN_FREQ:
          vocab[word] = len(vocab)

  idx2word = {idx: word for word, idx in vocab.items()}
  print("Vocabulary size:", len(vocab))
  with open(os.path.join(SAVE_DIR, "vocab.pkl"), "wb") as f:
    pickle.dump(vocab, f)
  return vocab


In [ ]:
DATASETS = {
    "flickr8k": {
        "ROOT": "/content/flickr8k",
        "IMAGE_DIR": "/content/flickr8k/Images",
        "CAPTION_FILE": "/content/flickr8k/captions.txt"
    },
    "flickr30k": {
        "ROOT": "/content/flickr30k",
        "IMAGE_DIR": "/content/flickr30k/Images",
        "CAPTION_FILE": "/content/flickr30k/captions.txt"
    }
}

for dataset_name, cfg in DATASETS.items():
    print(f"\nProcessing {dataset_name}")
    ROOT = cfg["ROOT"]
    IMAGE_DIR = cfg["IMAGE_DIR"]
    CAPTION_FILE = cfg["CAPTION_FILE"]

    print(f"Total images in {dataset_name}:", len(os.listdir(IMAGE_DIR)))
    df = pd.read_csv(CAPTION_FILE)
    print(df.head())
    print(df.columns)
    print(df.shape)

    # Fill NaN values in 'caption' column with empty strings and ensure string type
    df['caption'] = df['caption'].fillna('').astype(str)

    images = df["image"].unique().tolist()
    print(f"Unique images in {dataset_name}:", len(images))

    split_dict = split_dataset(images,dataset_name)

    vocab = create_vocab(df,split_dict,dataset_name)



Processing flickr8k
Total images in flickr8k: 8091
                       image  \
0  1000268201_693b08cb0e.jpg   
1  1000268201_693b08cb0e.jpg   
2  1000268201_693b08cb0e.jpg   
3  1000268201_693b08cb0e.jpg   
4  1000268201_693b08cb0e.jpg   

                                             caption  
0  A child in a pink dress is climbing up a set o...  
1              A girl going into a wooden building .  
2   A little girl climbing into a wooden playhouse .  
3  A little girl climbing the stairs to her playh...  
4  A little girl in a pink dress going into a woo...  
Index(['image', 'caption'], dtype='object')
(40455, 2)
Unique images in flickr8k: 8091
Train : 6068
Val   : 1011
Test  : 1012
Vocabulary size: 2573

Processing flickr30k
Total images in flickr30k: 31811
            image                                            caption
0  1000092795.jpg   Two young guys with shaggy hair look at their...
1  1000092795.jpg   Two young , White males are outside near many...
2  1000092795.j

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import shutil

folder_path = "/content/drive/MyDrive/MMRetrieval/flickr8k"
output_zip = "/content/drive/MyDrive/MMRetrieval/flickr8k"

shutil.make_archive(output_zip, 'zip', folder_path)

print("ZIP created:", output_zip + ".zip")

ZIP created: /content/drive/MyDrive/MMRetrieval/flickr8k.zip


In [17]:
folder_path = "/content/drive/MyDrive/MMRetrieval/flickr30k"
output_zip = "/content/drive/MyDrive/MMRetrieval/flickr30k"

shutil.make_archive(output_zip, 'zip', folder_path)

print("ZIP created:", output_zip + ".zip")

ZIP created: /content/drive/MyDrive/MMRetrieval/flickr30k.zip
